<a href="https://colab.research.google.com/github/itsrohan888888/projects-for-class-/blob/main/dlproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import zipfile
import os

zip_path = "/content/chromosome detection.v1-chromosome-detection-2.yolov8.zip"
extract_path = "/content/chromosome_dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print(os.listdir(extract_path))

Dataset extracted successfully!
['data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'valid', 'test', 'train']


In [4]:

import os

extract_path = "/content/chromosome_dataset"

print("Extracted files and folders:")
print(os.listdir(extract_path))

# Automatically locate data.yaml
DATASET_DIR = None

for root, dirs, files in os.walk(extract_path):
    if "data.yaml" in files:
        DATASET_DIR = root
        break

if DATASET_DIR:
    print("\nDataset found at:", DATASET_DIR)
    print("\nDataset contents:")
    print(os.listdir(DATASET_DIR))
else:
    print("\ndata.yaml not found. Check the extracted folder.")



Extracted files and folders:
['data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'valid', 'test', 'train']

Dataset found at: /content/chromosome_dataset

Dataset contents:
['data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'valid', 'test', 'train']


In [5]:

import os
import yaml

DATASET_DIR = "/content/chromosome_dataset"

# Read dataset configuration
with open(os.path.join(DATASET_DIR, "data.yaml"), "r") as f:
    config = yaml.safe_load(f)

print("========== DATASET CONFIGURATION ==========")
print(config)

# Inspect train, valid and test folders
for split in ["train", "valid", "test"]:
    split_path = os.path.join(DATASET_DIR, split)

    print(f"\n========== {split.upper()} ==========")

    if os.path.exists(split_path):
        print("Contents:", os.listdir(split_path))

        for subfolder in os.listdir(split_path):
            subpath = os.path.join(split_path, subfolder)

            if os.path.isdir(subpath):
                print(f"{subfolder}: {len(os.listdir(subpath))} files")
    else:
        print("Folder not found!")



========== DATASET CONFIGURATION ==========
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 25, 'names': ['A1', 'A2', 'A3', 'B4', 'B5', 'C10', 'C11', 'C12', 'C6', 'C7', 'C8', 'C9', 'D13', 'D14', 'D15', 'E16', 'E17', 'E18', 'F19', 'F20', 'G21', 'G22', 'X', 'Y', 'chromosomes'], 'roboflow': {'workspace': 'chromosomesdetection', 'project': 'chromosome-detection-dbsdf', 'version': 1, 'license': 'Public Domain', 'url': 'https://universe.roboflow.com/chromosomesdetection/chromosome-detection-dbsdf/dataset/1'}}

========== TRAIN ==========
Contents: ['images', 'labels']
images: 3607 files
labels: 3607 files

========== VALID ==========
Contents: ['images', 'labels']
images: 1017 files
labels: 1017 files

========== TEST ==========
Contents: ['images', 'labels']
images: 525 files
labels: 525 files


In [6]:

!pip install -q ultralytics



In [7]:

import os
import yaml

DATASET_DIR = "/content/chromosome_dataset"

data_config = {
    "path": DATASET_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 25,
    "names": [
        "A1", "A2", "A3", "B4", "B5",
        "C10", "C11", "C12", "C6", "C7",
        "C8", "C9", "D13", "D14", "D15",
        "E16", "E17", "E18", "F19", "F20",
        "G21", "G22", "X", "Y", "chromosomes"
    ]
}

fixed_yaml = "/content/chromosome_dataset_fixed.yaml"

with open(fixed_yaml, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("Configuration saved:", fixed_yaml)

for split in ["train", "valid", "test"]:
    image_dir = os.path.join(DATASET_DIR, split, "images")
    label_dir = os.path.join(DATASET_DIR, split, "labels")

    print(
        split,
        "| Images:", len(os.listdir(image_dir)),
        "| Labels:", len(os.listdir(label_dir))
    )


Configuration saved: /content/chromosome_dataset_fixed.yaml
train | Images: 3607 | Labels: 3607
valid | Images: 1017 | Labels: 1017
test | Images: 525 | Labels: 525


In [8]:
from ultralytics import YOLO
import torch

# Confirm GPU
assert torch.cuda.is_available(), "Enable GPU before training!"

print("Using:", torch.cuda.get_device_name(0))

# Lightweight YOLOv8 nano model
model = YOLO("yolov8n.pt")

results = model.train(
    data=fixed_yaml,

    epochs=3,
    imgsz=256,
    batch=32,

    device=0,
    workers=4,

    project="/content/drive/MyDrive/Chromosome_Project",
    name="ultra_fast",

    exist_ok=True,
    save=True,
    save_period=1,

    patience=3,
    cache=True,
    amp=True,

    plots=False,
    verbose=True
)

print("TRAINING COMPLETED!")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Using: Tesla T4
Ultralytics 8.4.193 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/chromosome_dataset_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torc

In [10]:
from ultralytics import YOLO

best_yolo = YOLO(
    "/content/drive/MyDrive/Chromosome_Project/ultra_fast/weights/best.pt"
)

print("YOLO model loaded successfully!")


YOLO model loaded successfully!


In [11]:
yolo_metrics = best_yolo.val(
    data=fixed_yaml,
    split="test",
    plots=True
)

print("\nYOLOv8 Test Results:")
print(yolo_metrics.results_dict)

Ultralytics 8.4.193 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,138,539 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1076.0±140.1 MB/s, size: 35.5 KB)
val: Scanning /content/chromosome_dataset/test/labels... 525 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 525/525 1.6Kit/s 0.3s
val: New cache created: /content/chromosome_dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 5.1it/s 6.4s
                   all        525      24125      0.194      0.383      0.159      0.107
                    A1        506       1012      0.201      0.708      0.324      0.213
                    A2        506       1012      0.246      0.676      0.315      0.221
                    A3        506       1012      0.142       0.64      0.162      0.108
                    B4        506       1011      0.242      0.5